<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ZeroToHeroColabCollection/blob/main/recall_nano_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Standard Imports
import re
import math
import torch
import random
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [ ]:
# Hyper Parameters
BATCH_SIZE = 32
CONTEXT_LENGTH = 128
N_EMBD = 256
N_HEAD = 4
N_BLOCKS = 6
TRAIN_TEST_RATIO = 0.8
LR = 4e-4

In [ ]:
# Dataset Retrieval
with open('goft.txt', 'r') as file:
  text = file.read()
  text = re.sub('[{}|-]', '', text)

In [ ]:
# Character to Integer Mapping Table
chars = sorted(list(set(text)))
stoi = {char: i for i, char in enumerate(chars)}
itos = dict(zip(stoi.values(), stoi.keys()))
''.join(chars), len(chars)

('\t\n !"&\'(),.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXY[]abcdefghijklmnopqrstuvwxyz',
 77)

In [ ]:
# Encoder and Decoder Functions
encode = lambda string: list(map(lambda char: stoi[char], list(string)))
decode = lambda ixs: ''.join(list(map(lambda ix: itos[ix], ixs)))

In [ ]:
# Encoding Our Dataset
nfill = int((1 - (len(text)/CONTEXT_LENGTH - len(text)//CONTEXT_LENGTH)) * CONTEXT_LENGTH)
tokens = torch.tensor(encode(text) + [0] * nfill, dtype = torch.long)
tokens_shift = torch.cat((tokens[1:], torch.zeros(1, dtype=torch.long)), dim=-1)
tokens, tokens_shift = tokens.view(-1, CONTEXT_LENGTH), tokens_shift.view(-1, CONTEXT_LENGTH)
tokens.shape, tokens_shift.shape

(torch.Size([12632, 128]), torch.Size([12632, 128]))

In [ ]:
# Training Test Data Split
n = int(TRAIN_TEST_RATIO * tokens.shape[0])
Xtr, Ytr = tokens[:n], tokens_shift[:n]
Xte, Yte = tokens[n:], tokens_shift[n:]
Ytr.shape, Yte.shape

(torch.Size([10105, 128]), torch.Size([2527, 128]))

In [ ]:
# Mini-Batching the Data
def get_batch(data, targets, batch_size):
  ix = torch.randint(0, data.shape[0], size = (batch_size,))
  return data[ix], targets[ix]

In [ ]:
# Attention Head
class Head(nn.Module):
  def __init__(self, head_size, n_embd, context_length):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size)
    self.query = nn.Linear(n_embd, head_size)
    self.value = nn.Linear(n_embd, head_size)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

  def forward(self, x):
    k, q, v = self.key(x), self.query(x), self.value(x) # B, T, head_size
    wei = k @ q.transpose(-2, -1) # B, T, head_size @ B, head_size, T = B, T, T
    wei = torch.masked_fill(wei, self.tril == 0, float('-inf'))
    wei = F.softmax(wei, dim = -1)
    out = wei @ v # B, T, T @ B, T, head_size
    return out

In [ ]:
# Multi Head Attention
class MultiHeadAttention(nn.Module):
  def __init__ (self, n_embd, context_length, n_heads):
    super().__init__()
    self.n_heads = n_heads
    self.key = nn.Linear(n_embd, n_embd)
    self.query = nn.Linear(n_embd, n_embd)
    self.value = nn.Linear(n_embd, n_embd)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

  def forward(self, x):
    B, T, C = x.shape
    # print(f"{x.shape=}")
    k, q, v = self.key(x), self.query(x), self.value(x)
    # print(f"{k.shape=}")
    k, q, v = k.view(B, T, self.n_heads, -1).transpose(1, 2), q.view(B, T, self.n_heads, -1).transpose(1, 2), v.view(B, T, self.n_heads, -1).transpose(1, 2)
    # print(f"{k.shape=}")
    wei = q @ k.transpose(-1, -2)
    # print(f"{wei.shape=}")
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim = -1) # Still not sure y d last dim
    # print(f"{wei.shape=}, {v.shape=}")
    out = wei @ v
    # print(f"{out.shape=}")
    out = out.transpose(1, 2)
    # print(f"{out.shape=}")
    out = out.reshape(B, T, C)
    # print(f"{out.shape=}")
    return out


In [ ]:
# Feed Forward Network
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.neural_net = nn.Sequential(
        nn.Linear(n_embd, n_embd),
        nn.ReLU()
    )

  def forward(self, x):
    out = self.neural_net(x) + x
    return out

In [ ]:
# Building Blocks Now
class Block(nn.Module):
  def __init__(self, n_blocks, n_heads, n_embd, batch_size, context_length):
    super().__init__()
    self.block = nn.Sequential(MultiHeadAttention(n_heads=n_heads, n_embd=n_embd, context_length=context_length), FeedForward(n_embd))
    self.ln = nn.LayerNorm(normalized_shape=(n_embd))

  def forward(self, x):
    out = self.block(x) + x
    out = self.ln(x) + out
    return out

In [ ]:
class GPT(nn.Module):
  def __init__(self, vocab_size, batch_size, n_embd, context_length, n_heads, n_blocks):
    super().__init__()
    self.context_length = context_length
    self.vocab_embedding = nn.Embedding(vocab_size, n_embd)
    self.pos_embedding = nn.Embedding(context_length, n_embd)
    self.blocks = nn.Sequential(*[Block(n_blocks, n_heads, n_embd, batch_size, context_length) for _ in range(n_blocks)])
    self.ln_f = nn.LayerNorm((n_embd))
    self.lh = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets=None):
    emb, pos = self.vocab_embedding(x), self.pos_embedding(torch.arange(x.shape[1])) # B, T, n_embd
    prelogits = self.blocks(emb + pos)
    prenorm_logits = self.ln_f(prelogits)
    logits = self.lh(prenorm_logits) # B, T, vocab_size
    if targets is None:
      return logits
    else:
      loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
      return logits, loss

  def smpl(self, start_tokens, max_output_tokens):
    idx = start_tokens
    for i in range(max_output_tokens):
      logits = self(idx[:, -self.context_length:]) # 1, T, vocab_size
      probs = F.softmax(logits.view(-1, logits.shape[-1]), dim = -1)
      char = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, char[-1].view(-1, 1)), dim = - 1)
    return idx

In [ ]:
gpt = GPT(len(chars), BATCH_SIZE, N_EMBD, CONTEXT_LENGTH, N_HEAD, N_BLOCKS)
logits, loss = gpt(*get_batch(Xtr, Ytr, BATCH_SIZE))
loss

tensor(4.5037, grad_fn=<NllLossBackward0>)

In [ ]:
logits.shape

torch.Size([32, 128, 77])

In [ ]:
# Optmizier
optmizier = torch.optim.AdamW(gpt.parameters(), lr = LR)

In [ ]:
print(sum(p.numel() for p in gpt.parameters()))

1654861


In [ ]:
print(decode(gpt.smpl(torch.zeros(1,1, dtype = torch.long), 1000).tolist()[0]))

	o7DO0G8'u,:IVv
hcwd7jwK(MW7bs3YP1ptzvo;MIC :
bNzK6S'L?(T5UE'()W0YzECw5msFCm]AUbvrOiH7r6ErcN]amMl!;7U)pGcaW pJhFMg!G[zP
gxCrm4Y80A,Kh[Lj6V?xDf	aXaJbLkRvw5zr;v?j!?4M:jrvn1aG)52W4jLD 0 g5Ss.pQpT]x:Da9ov13nmHOQH19x,!2aI	(NHp0g"
U1bBc!WhLJ? Ka1WDQ5" DsL:vw6U6J'QI?T.60!'rDgN,GmhWLN2I9FgdLOKWn,Wwa3r	&
6QH?bUKpQ&M01HOJG	CjVkx?S,"9xn]Qu'VwnVd	9h8VThV6"Xl.r.v2]p4sb5O.3a7A4cf:mbxJw(,:I0Ix7xEz! n?swpqU(bRzal28tU	oJhS	G!LL1.uQT?gKNV;PQy!(:wotTmDEPmwaKVoB.!g"fXTJWSACBwUA 4taWwVWTJ,LrQ?E5[S.ajUf"ENrSPDR?B6B Q9R]yH2vhR:)0s,NO1,J)t [uAU,S!Si:SQL0BcD5cVQMNrdr1SwzAX[(l!jLV81oV&(Bla S1m!Qgwv	Sh'VU Q0?vlF0Ovl4819P:VVxOTg]"F9Vw:PK0bpg:?EQ?:&;fMJ db:rG)rmRG!H"
Qs"OlVJU	xC2" 
4.OULtyH&VgpDt	MDmxN3cXEr5bl5Mm?K]MyFQh	)6.J(jmmdOQ5)4 :LPcM3H(	UHK'W:LU5MMwS4	b:w!V4E,WGaSveQz6y34'COM]	RAlDq5)HTMpHChOU".FtIOY8C9vQX4(!al4!
CR	95,RJ(dHfBC)a:3l4PpNruT u'JN.!4e[2ksbyUD!UJj!QaV[RN	wf'C;0R gyrHbadj,	!?m[g.:Q E	nsGcie[SaCdR	8RmHyv0cVurah:)tNbn16a5,mI	DOA!uN.L&W?wJ5OD:'n;3e2k? epcoAyk!"D;D!(H&gn"KFXS5p:4fHhjfE2Lp9uUIrFmfK5

In [ ]:
a = nn.Linear(256, 256)

In [ ]:
a(torch.zeros(32, 128, 256)).shape